# Day 13 — Professional Report Generation & Venture Copilot (Layer 13) 🎉

The **final layer** of Architecture V1.0 — the face of the platform. It turns every prior layer's
outputs into audience-specific, decision-ready documents: a **Founder Intelligence Report**, an
**Investor Due Diligence Report**, an **Investment Memo**, an **Executive Summary**, and a
conversational **Venture Copilot** — exported to JSON / Markdown / HTML.

**Honesty — the defining principle.** Reporting generates **zero new analysis**: every report is a
deterministic *reformatting* of the Day 4-12 outputs. So the honesty is **inherited** — the one rule
is **NO NEW CLAIMS**: every section names its `source` (provenance), and anything the data can't
support (founder background, revenue, moat, expected return) goes in a **deferred appendix**, never
fabricated. Deterministic, **no LLM**, no faiss. Logic in `src/reporting/`.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import pandas as pd, numpy as np

# One warm faiss-free stack: build the ReportBundle ONCE -> all 4 reports + copilot.
from src.reporting.pipeline import ReportingPipeline
pipe = ReportingPipeline()

df = pd.read_csv("data/processed/kickstarter_processed.csv"); ti = np.load("artifacts/splits/test_index.npy")
row = df.iloc[ti][df.iloc[ti]["usd_goal_real"] > 12000].sample(1, random_state=4).iloc[0].to_dict()
out = pipe.run(row, export=True)                 # 4 reports + copilot + multi-format export, one bundle

def show(report, ids=None):
    for s in report["sections"]:
        if ids and s["id"] not in ids: continue
        print("\n## %s   [source: %s]" % (s["title"], s["source"]))
        b = s["body"]
        if s["kind"] == "text": print("  ", b)
        elif s["kind"] == "metrics":
            for k, v in b.items(): print("   %-34s %s" % (k, v))
        elif s["kind"] == "list":
            for it in b: print("   -", it)
        elif s["kind"] == "table":
            print("   ", " | ".join(b["columns"]))
            for r in b["rows"][:4]: print("   ", " | ".join(str(x)[:34] for x in r))

print("CAMPAIGN:", out["campaign"]["name"])
print("reports :", list(out["reports"].keys()), "| copilot turns:", len(out["copilot"]["turns"]))

CAMPAIGN: Artificer - Board Game
reports : ['founder', 'investor', 'memo', 'executive'] | copilot turns: 8


## Founder Intelligence Report

Actionable and encouraging — where you stand, strengths, fixable weaknesses, and a prioritized to-do list. Every section sourced.

In [2]:
show(out["reports"]["founder"], ids={"scorecard","strengths","weaknesses","priority_improvements","next_milestones"})


## Scorecard   [source: Days 4 / 10 / 11]
   Startup score (readiness index)    45
   Funding probability                24%
   Investment readiness               Fixable (reachable 66%)
   Confidence                         83%

## Strengths   [source: Day-5 SHAP (positive drivers)]
   - your category's overall popularity works in your favor.
   - being in the Games category works in your favor.
   - your title's capitalization style works in your favor.
   - title term 'board game' works in your favor.

## Weaknesses   [source: Day-5 SHAP (negative drivers)]
   - your funding goal size weighs against you.
   - title term 'game' weighs against you.
   - your goal relative to the campaign length weighs against you.
   - your title length weighs against you.

## Priority Improvements   [source: Day-11 prioritization (Day-7 evidence)]
    # | Action | Impact | Evidence
    1 | Shorten your campaign from 29 to 1 | +30 pts | Games campaigns with a <=15d campa
    2 | Lower your funding go

## Investor Due Diligence Report

Sceptical — the committee recommendation and the six specialist agent verdicts it was computed from (no re-analysis, so the body can't contradict the headline).

In [3]:
iv = out["reports"]["investor"]
print("RECOMMENDATION:", iv["recommendation"], "| risk veto:", iv["risk_veto"])
show(iv, ids={"investment_recommendation","market_analysis","financial_assessment","risks_and_red_flags","investment_decision"})

RECOMMENDATION: Do Not Invest | risk veto: {'eligible': True, 'applied': False}

## Investment Recommendation   [source: Day-12 committee]
   Recommendation                     Do Not Invest
   Overall confidence                 78%
   Agreement                          83%
   Committee score                    -0.69

## Market Analysis   [source: Day-12 market agent]
   - Verdict: positive (score 72/100, confidence 87%).
   - Games campaigns fund at 44% vs a 40% platform baseline (lift 1.09).
   - Market timing is favorable; the Games trend is rising (22,878 campaigns).
   - The category is crowded.

## Financial Assessment (funding viability)   [source: Day-12 financial agent]
   - Verdict: negative (score 24/100, confidence 78%).
   - Goal $30,516 is 6.1x the category winners' median ($5,000).
   - Calibrated funding probability 24% vs the 33% decision bar.
   - Goal-size risk (High): Goal $30,516 is 6.1x the Games funded-median ($5,000); large goals are harder to reach.

## Risks &

## Investment Memo

VC-style narrative culminating in a deterministic **investment thesis** (bull = strongest agent, bear = weakest, conclusion = readiness stage).

In [4]:
show(out["reports"]["memo"], ids={"investment_thesis","recommendation"})


## Investment Thesis   [source: Day-12 committee + Days 10/11 synthesis]
   The committee recommends Do Not Invest at 78% confidence. Bull case (Market): Games campaigns fund at 44% vs a 40% platform baseline (lift 1.09). Bear case (Competition): Ask for less: funded campaigns' median goal is $5,000 vs $10,000 for those that failed. As presented the campaign is not fundable (24%), but it is fixable — reachable to ~66% with the priority pre-launch changes. Conditional on those fixes the thesis could clear the bar; unchanged, pass.

## Recommendation   [source: Day-12 committee]
   Recommendation                     Do Not Invest
   Overall confidence                 78%
   Committee score                    -0.69
   Risk veto applied                  False


## Executive Summary

The five decision-sufficient numbers for a dashboard.

In [5]:
print(out["reports"]["executive"]["headline"], "\n")
show(out["reports"]["executive"], ids={"dashboard"})

Artificer - Board Game: Do Not Invest — Fixable at 24% (readiness 45), 78% confidence. 


## Dashboard   [source: Days 4 / 10 / 11 / 12]
   Startup score                      45
   Funding probability                24%
   Risk                               High
   Recommendation                     Do Not Invest
   Confidence                         78%


## Venture Copilot (deterministic Q&A — no LLM)

Intent-routing over the same bundle, with real numbers and provenance. Out-of-scope questions
(MRR, revenue, …) hit an **honest "not available" fallback** — the copilot never invents an answer.

In [6]:
for t in out["copilot"]["turns"]:
    tag = "DEFERRED" if t["deferred"] else t["intent"]
    print("Q: %s   [%s]" % (t["question"], tag))
    print("A: %s\n" % t["answer"][:160])

Q: Why is my funding probability low?   [probability]
A: Your funding probability is 24% (Fixable). With the priority pre-launch changes it's reachable to ~66%. Confidence 83%.

Q: What should I improve first?   [improve]
A: First: Shorten your campaign from 29 to 15 days — the model estimates about +30 points. Games campaigns with a <=15d campaign funded at 66% vs a 40% baseline (l

Q: Which campaigns are most similar to mine?   [similar]
A: The closest comparison is the 22,878 other Games campaigns in our data, which fund at 44% (vs a 40% baseline). You're positioned like the strugglers.

Q: Why do similar campaigns fail?   [why_fail]
A: Ask for less: funded campaigns' median goal is $5,000 vs $10,000 for those that failed. Your $30,516 goal is 6.1x the winners' median ($5,000) — priced like the

Q: What is my biggest risk?   [risk]
A: Your biggest risk: Goal $30,516 is 6.1x the Games funded-median ($5,000); large goals are harder to reach. (Overall risk: High).

Q: What should I do 

## Multi-format export

One report dict -> JSON / Markdown / HTML (deps-free); PDF is optional and skipped gracefully if no PDF library is installed (the HTML is print-ready).

In [7]:
for name, paths in out["exported"].items():
    print("%-10s ->" % name, list(paths.keys()))
print("\nPDF gracefully skipped (no new dependency):", all("pdf" not in p for p in out["exported"].values()))
print("HTML sample path:", out["exported"]["founder"]["html"])

founder    -> ['json', 'markdown', 'html']
investor   -> ['json', 'markdown', 'html']
memo       -> ['json', 'markdown', 'html']
executive  -> ['json', 'markdown', 'html']

PDF gracefully skipped (no new dependency): True
HTML sample path: outputs/reports/founder_intelligence_report__artificer_board_game.html


## Provenance & the honesty boundary

Every section names its source (the NO-NEW-CLAIMS rule made checkable), and each report appends
exactly what it can't assess — deferred, not fabricated.

In [8]:
fr = out["reports"]["founder"]
print("Founder-report section provenance:")
for s in fr["sections"]: print("   %-22s <- %s" % (s["id"], s["source"]))
print("\nEvery section provenance-tagged:", all(s["source"] for s in fr["sections"]))
print("\nDeferred appendix (%d items — no data, not fabricated):" % len(fr["deferred_appendix"]["items"]))
print("  ", ", ".join(fr["deferred_appendix"]["items"][:10]), "...")
# consistency: reports are projections of ONE bundle -> they agree
ex = out["reports"]["executive"]
score_f = next(s["body"]["Startup score (readiness index)"] for s in fr["sections"] if s["id"]=="scorecard")
score_e = next(s["body"]["Startup score"] for s in ex["sections"] if s["id"]=="dashboard")
print("\nfounder & executive agree on startup score (same bundle):", score_f == score_e, "(%s)" % score_f)

Founder-report section provenance:
   executive_summary      <- Day-11 synthesis
   scorecard              <- Days 4 / 10 / 11
   strengths              <- Day-5 SHAP (positive drivers)
   weaknesses             <- Day-5 SHAP (negative drivers)
   priority_improvements  <- Day-11 prioritization (Day-7 evidence)
   evolution_roadmap      <- Day-10 preparation roadmap
   scenario_analysis      <- Day-10 scenario simulator
   next_milestones        <- Day-10 milestones

Every section provenance-tagged: True

Deferred appendix (37 items — no data, not fabricated):
   Financial Strategy, Go-To-Market Strategy, Product Strategy, Team Strategy, Technology Strategy, build_partnerships, cap_table, competitive_moat, customer_retention, delay_fundraising_timing ...

founder & executive agree on startup score (same bundle): True (45)


## Serialization (content-hash identity + verify)

The reporting engine is a policy over the Day-4/11/12 stack. Report artifacts carry a timestamp, so they're hashed on **content with the timestamp excluded** — regenerating a report is not drift.

In [9]:
from src.reporting.serialization import ReportingSerializer
ser = ReportingSerializer().build(); ser.save()
v = ser.verify()
print(ser.describe())
print("verify all_ok:", v["all_ok"], "(%d/%d checks pass)" % (sum(v["checks"].values()), len(v["checks"])))

Reporting engine v1.0.0 (content 222007d6df27…): layer 13; 4 report types, 3 deps hashed, 5 artifacts (content, ts-excluded), 19 items deferred.
verify all_ok: True (9/9 checks pass)


## Day 13 complete — Layer 13 ✅  →  Architecture V1.0 functionally complete 🎉

The platform now turns a raw campaign into a full, coherent report suite — founder, investor, memo,
executive, and a conversational copilot — on one warm faiss-free stack, exported to multiple formats,
content-hash verified. Because every report is a pure projection of one shared bundle, they **cannot
contradict each other**, and because reporting adds no analysis, the honesty is **inherited**: every
section is sourced, and founder background / revenue / moat / expected return remain **deferred, not
invented**. Deterministic, no LLM. Model-simulated, not causal.

**Architecture V1.0 — all 13 layers complete.** From raw Kickstarter input → prediction →
explainability → knowledge → patterns → counterfactuals → objections → evolution → scenarios →
recommendations → multi-agent committee → professional reports. 🎉